# 02 — Run the final experiments

Attach two private Kaggle Datasets:

1. `dtp_fact_kaggle_dataset_2026-10-25.zip` (background Python package), and
2. an audited `dtp_fact_final_data_<date>.zip` following the included layout.

Enable a Kaggle GPU before running. This notebook invokes the reusable
command-line pipeline; no trainer or metric is implemented only here.
It writes resumable outputs to `/kaggle/working/final_run`.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile

ON_KAGGLE = Path("/kaggle/input").exists()
INPUT_ROOT = Path("/kaggle/input") if ON_KAGGLE else Path.cwd()
WORK_ROOT = Path("/kaggle/working/dtp_fact_runtime") if ON_KAGGLE else Path.cwd() / "_dtp_fact_runtime"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def find_background():
    candidates = sorted(INPUT_ROOT.rglob("background/pyproject.toml"))
    candidates += sorted(WORK_ROOT.rglob("background/pyproject.toml"))
    if not candidates:
        archives = sorted(INPUT_ROOT.rglob("dtp_fact_kaggle_dataset*.zip"))
        if not archives:
            raise FileNotFoundError(
                "Attach dtp_fact_kaggle_dataset_2026-10-25.zip as a Kaggle Dataset."
            )
        extract_root = WORK_ROOT / "dataset"
        extract_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            archive.extractall(extract_root)
        candidates = sorted(extract_root.rglob("background/pyproject.toml"))
    if not candidates:
        raise FileNotFoundError("The attached dataset has no background/pyproject.toml")
    return candidates[0].parent, candidates[0].parent.parent

CODE_ROOT, BUNDLE_ROOT = find_background()
DATA_ROOT = BUNDLE_ROOT / "data"
sys.path.insert(0, str(CODE_ROOT / "src"))

def install_background():
    install_command = [
        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
        "-e", f"{CODE_ROOT}[analysis,test,retrieval,ml]",
    ]
    subprocess.run(install_command, check=True)
    print("Background package installed from", CODE_ROOT)

print(json.dumps({
    "on_kaggle": ON_KAGGLE,
    "code_root": str(CODE_ROOT),
    "data_root": str(DATA_ROOT),
    "work_root": str(WORK_ROOT),
}, indent=2))


{
  "on_kaggle": false,
  "code_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\background",
  "data_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime\\dataset\\dtp_fact_kaggle_dataset\\data",
  "work_root": "D:\\1.msc\\ET2\\kaggle\\build-2026-10-25\\notebook-run-v1\\_dtp_fact_runtime"
}


In [2]:
def extract_named_archives(pattern, destination):
    destination.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(INPUT_ROOT.rglob(pattern)):
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(destination / archive_path.stem)

def find_final_data_root():
    search_roots = [INPUT_ROOT, WORK_ROOT]
    manifests = []
    for root in search_roots:
        manifests.extend(root.rglob("frozen/dataset_manifest.json"))
    if not manifests:
        extract_named_archives("dtp_fact_final_data*.zip", WORK_ROOT / "final_inputs")
        manifests = list((WORK_ROOT / "final_inputs").rglob("frozen/dataset_manifest.json"))
    return manifests[0].parent.parent if manifests else None

FINAL_DATA_ROOT = find_final_data_root()
print("Final data root:", FINAL_DATA_ROOT or "NOT FOUND")


Final data root: NOT FOUND


In [3]:
readiness = {"status": "checking", "blockers": []}
if FINAL_DATA_ROOT is None:
    readiness["blockers"].append("Audited dtp_fact_final_data_<date>.zip is not attached.")
else:
    required = [
        FINAL_DATA_ROOT / "frozen/dataset_manifest.json",
        FINAL_DATA_ROOT / "frozen/claims.jsonl",
        FINAL_DATA_ROOT / "frozen/evidence.jsonl",
        FINAL_DATA_ROOT / "image_manifest.csv",
        FINAL_DATA_ROOT / "conflict_donors.jsonl",
    ]
    readiness["blockers"].extend(f"Missing {path}" for path in required if not path.is_file())
    manifest_path = FINAL_DATA_ROOT / "frozen/dataset_manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("status") != "frozen_dataset":
            readiness["blockers"].append("dataset_manifest.json is not a frozen_dataset manifest.")
try:
    import torch
    if not torch.cuda.is_available():
        readiness["blockers"].append("No CUDA GPU is enabled in this Kaggle session.")
    readiness["cuda_devices"] = torch.cuda.device_count()
    readiness["cuda_name"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except ImportError:
    readiness["blockers"].append("PyTorch is unavailable.")
readiness["status"] = "ready_for_final_run" if not readiness["blockers"] else "blocked"
readiness_path = WORK_ROOT / "02_final_run_readiness.json"
readiness_path.write_text(json.dumps(readiness, indent=2) + "\n", encoding="utf-8")
print(json.dumps(readiness, indent=2))


{
  "status": "blocked",
  "blockers": [
    "Audited dtp_fact_final_data_<date>.zip is not attached.",
    "No CUDA GPU is enabled in this Kaggle session."
  ],
  "cuda_devices": 0,
  "cuda_name": null
}


In [4]:
if readiness["status"] == "ready_for_final_run":
    install_background()
else:
    print("Installation and training skipped because final-run prerequisites are incomplete.")


Installation and training skipped because final-run prerequisites are incomplete.


In [5]:
RESULTS_ROOT = Path("/kaggle/working/final_run") if ON_KAGGLE else WORK_ROOT / "final_run"
TABLE_ROOT = Path("/kaggle/working/generated_tables") if ON_KAGGLE else WORK_ROOT / "generated_tables"
validation_report = None

if readiness["status"] == "ready_for_final_run":
    command = [
        sys.executable, "scripts/run_full_study.py",
        "--config", "configs/base.yaml",
        "--frozen-data-directory", str(FINAL_DATA_ROOT / "frozen"),
        "--claim-image-manifest", str(FINAL_DATA_ROOT / "image_manifest.csv"),
        "--conflict-manifest", str(FINAL_DATA_ROOT / "conflict_donors.jsonl"),
        "--output-directory", str(RESULTS_ROOT),
        "--paper-table-directory", str(TABLE_ROOT),
        "--device", "cuda",
        "--resume",
    ]
    optional = {
        "--evidence-image-manifest": FINAL_DATA_ROOT / "evidence_image_manifest.csv",
        "--evidence-text": FINAL_DATA_ROOT / "evidence_text.jsonl",
    }
    for flag, path in optional.items():
        if path.is_file():
            command.extend([flag, str(path)])
    print("Running:", " ".join(command))
    subprocess.run(command, cwd=CODE_ROOT, check=True)

    from evaluation.study_validation import validate_study_artifacts
    validation_report = validate_study_artifacts(
        CODE_ROOT / "configs/base.yaml",
        FINAL_DATA_ROOT / "frozen",
        RESULTS_ROOT,
        TABLE_ROOT,
    )
    validation_path = RESULTS_ROOT / "study_validation_report.json"
    validation_path.write_text(json.dumps(validation_report, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    print(json.dumps({
        "status": validation_report["status"],
        "scientific_claims_allowed": validation_report["scientific_claims_allowed"],
        "blockers": validation_report["blockers"],
    }, indent=2))
    shutil.make_archive(str(Path("/kaggle/working/dtp_fact_final_results") if ON_KAGGLE else WORK_ROOT / "dtp_fact_final_results"), "zip", RESULTS_ROOT.parent, RESULTS_ROOT.name)
    shutil.make_archive(str(Path("/kaggle/working/dtp_fact_generated_tables") if ON_KAGGLE else WORK_ROOT / "dtp_fact_generated_tables"), "zip", TABLE_ROOT.parent, TABLE_ROOT.name)
else:
    blocked = {
        "status": "blocked_missing_audited_data_or_gpu",
        "scientific_claims_allowed": False,
        "blockers": readiness["blockers"],
    }
    (WORK_ROOT / "02_experiment_status.json").write_text(json.dumps(blocked, indent=2) + "\n", encoding="utf-8")
    print("No scientific experiment was run.")


No scientific experiment was run.


A completed run creates `dtp_fact_final_results.zip` and
`dtp_fact_generated_tables.zip` in Kaggle's Output pane. Save a Kaggle
version so notebook 03 can attach those outputs. If this notebook reports
`blocked`, do not interpret that as a negative study result—the final data
or GPU prerequisite is simply absent.
